# AIMO Progress Prize 3 — TIR Baseline Pipeline

**Approach**: Tool-Integrated Reasoning (TIR) + Self-Correction + Majority Voting

- Engine: `transformers` + `bitsandbytes` (pre-installed on Kaggle, no internet needed)
- Inference: `NUM_SAMPLES` independent samples → majority vote
- TIR: LLM writes Python → sandbox executes → output fed back → self-correct
- Chat template applied properly for DeepSeek-R1 models
- `<think>` tag aware answer extraction

In [ ]:
# ============================================================
# Cell 1: Imports & Environment Detection
# ============================================================
import os
import re
import sys
import time
import tempfile
import subprocess
from pathlib import Path
from collections import Counter
from typing import List, Optional, Dict

import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local (mock mode)'}")
print(f"Python      : {sys.version.split()[0]}")

if IS_KAGGLE:
    import torch
    print(f"PyTorch     : {torch.__version__}")
    print(f"CUDA avail  : {torch.cuda.is_available()}"
          + (f" | {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ''))


In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================

# --- Model ---
# Add your Kaggle dataset slug here.
# e.g. if your dataset URL is kaggle.com/datasets/yourname/deepseek-r1-distill-qwen-7b
# then MODEL_DATASET = 'yourname/deepseek-r1-distill-qwen-7b'
MODEL_DATASET   = 'deepseek-r1-distill-qwen-7b'
MODEL_PATH      = f'/kaggle/input/{MODEL_DATASET}'
LOAD_IN_4BIT    = True    # INT4 via bitsandbytes; keeps 7B within 16 GB P100

# --- Context ---
MAX_MODEL_LEN   = 8192    # prompt + generation hard cap

# --- Sampling ---
NUM_SAMPLES     = 8 if IS_KAGGLE else 4    # maj@N
TEMPERATURE     = 0.6
TOP_P           = 0.95
MAX_NEW_TOKENS  = 4096

# --- TIR ---
MAX_TIR_ROUNDS  = 3
CODE_TIMEOUT    = 30      # seconds per code block execution

# --- I/O ---
TEST_CSV   = ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv'
              if IS_KAGGLE else 'dummy_test.csv')
OUTPUT_CSV = 'submission.csv'

print('Configuration:')
for k, v in {
    'MODEL_PATH': MODEL_PATH, 'LOAD_IN_4BIT': LOAD_IN_4BIT,
    'NUM_SAMPLES': NUM_SAMPLES, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS,
    'MAX_TIR_ROUNDS': MAX_TIR_ROUNDS, 'CODE_TIMEOUT': CODE_TIMEOUT,
}.items():
    print(f'  {k:20s}: {v}')


In [ ]:
# ============================================================
# Cell 3: Python Sandbox
# ============================================================

def execute_python(code: str, timeout: int = CODE_TIMEOUT) -> str:
    """Run code in isolated subprocess; return stdout+stderr (max 2000 chars)."""
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='.py', delete=False, encoding='utf-8'
    ) as f:
        f.write(code)
        tmp_path = f.name
    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True, text=True, timeout=timeout,
        )
        output = (proc.stdout + proc.stderr).strip() or '(no output)'
    except subprocess.TimeoutExpired:
        output = f'TimeoutError: exceeded {timeout}s'
    except Exception as exc:
        output = f'ExecutionError: {exc}'
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
    if len(output) > 2000:
        output = output[:1500] + '\n...[truncated]...\n' + output[-500:]
    return output


_t = execute_python('print(sum(range(1,101)))')
assert _t.strip() == '5050', f'Sandbox failed: {_t!r}'
print(f'Sandbox OK -> {_t}')


In [ ]:
# ============================================================
# Cell 4: Prompt Templates & Regex Helpers
# ============================================================

SYSTEM_PROMPT = (
    'You are an expert mathematician solving competition math problems.\n'
    'Reason step by step inside <think>...</think> tags.\n'
    'You may write Python code to assist with calculations.\n'
    '\n'
    'Wrap Python code in triple backticks:\n'
    '```python\n'
    '# your code here\n'
    '```\n'
    'The code output will be shown to you. Use code for arithmetic, modular arithmetic,\n'
    'combinatorics, exhaustive search, or verification.\n'
    '\n'
    'After reasoning, state your final integer answer EXACTLY as:\n'
    'The answer is \\boxed{<integer>}'
)

_CODE_RE  = re.compile(r'```python\s*(.*?)```', re.DOTALL)
_BOXED_RE = re.compile(r'\\boxed\{([^}]+)\}')


def extract_code_blocks(text: str) -> List[str]:
    return [b.strip() for b in _CODE_RE.findall(text)]


def extract_final_answer(text: str) -> Optional[str]:
    """
    Extract the last \\boxed{N} from the model output.
    Prefers the answer AFTER </think> to avoid extracting
    intermediate answers from the thinking block.
    """
    # First try: look only after </think>
    think_end = text.rfind('</think>')
    if think_end != -1:
        post = text[think_end:]
        m = _BOXED_RE.findall(post)
        if m:
            return m[-1].strip()
    # Fallback: search the entire text
    m = _BOXED_RE.findall(text)
    return m[-1].strip() if m else None


def build_messages(problem: str) -> List[Dict]:
    """Build a chat messages list for the given problem."""
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Problem: {problem}'},
    ]


# Sanity checks
assert extract_final_answer('The answer is \\boxed{42}') == '42'
assert extract_final_answer('<think>\\boxed{99}</think>\\boxed{42}') == '42', \
    'Should prefer answer after </think>'
assert extract_code_blocks('```python\nprint(1)\n```') == ['print(1)']
print('Prompt helpers OK')


In [ ]:
# ============================================================
# Cell 5: Model Loading
#   Kaggle : HuggingFace transformers + bitsandbytes
#   Local  : MockLLM (no GPU required)
# ============================================================

class _CompletionOutput:
    def __init__(self, text: str): self.text = text

class _RequestOutput:
    def __init__(self, texts: List[str]):
        self.outputs = [_CompletionOutput(t) for t in texts]

class SamplingParams:
    def __init__(self, temperature=0.6, top_p=0.95, max_tokens=4096, n=1):
        self.temperature = temperature
        self.top_p       = top_p
        self.max_tokens  = max_tokens
        self.n           = n


if IS_KAGGLE:
    # ---- Real inference via transformers + bitsandbytes ----
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(
            f'Model not found at {MODEL_PATH}.\n'
            f'Add the dataset to kernel-metadata.json -> dataset_sources.'
        )

    print('Loading tokenizer ...')
    _tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token

    print(f'Loading model (4-bit={LOAD_IN_4BIT}) ...')
    _mkw = dict(device_map='auto', trust_remote_code=True)
    if LOAD_IN_4BIT:
        _mkw['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
    else:
        _mkw['torch_dtype'] = torch.float16

    _hf = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **_mkw)
    _hf.eval()
    print('Model ready.')

    class TransformersLLM:
        """Wraps HF model with the same .generate() interface as MockLLM."""

        def format_prompt(self, messages: List[Dict]) -> str:
            """Apply the model's chat template to a messages list."""
            return _tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        def generate(self, prompts: List[str], sp: SamplingParams) -> List[_RequestOutput]:
            results = []
            for prompt in prompts:
                enc = _tok(
                    prompt, return_tensors='pt', truncation=True,
                    max_length=MAX_MODEL_LEN - sp.max_tokens,
                ).to(_hf.device)

                with torch.no_grad():
                    out_ids = _hf.generate(
                        **enc,
                        max_new_tokens=sp.max_tokens,
                        do_sample=(sp.temperature > 0),
                        temperature=max(sp.temperature, 1e-6),
                        top_p=sp.top_p,
                        num_return_sequences=sp.n,
                        pad_token_id=_tok.eos_token_id,
                        eos_token_id=_tok.eos_token_id,
                    )

                in_len = enc.input_ids.shape[1]
                texts  = [_tok.decode(ids[in_len:], skip_special_tokens=True)
                          for ids in out_ids]
                results.append(_RequestOutput(texts))
            return results

    llm = TransformersLLM()

else:
    # ---- MockLLM for local testing (no GPU) ----
    print('Local mode: using MockLLM')

    _MOCKS = [
        ('<think>Let me compute.</think>\n'
         '```python\nprint(sum(range(1,101)))\n```\n'
         'The answer is \\boxed{5050}'),
        ('<think>Writing code.</think>\n'
         '```python\nprint(100*101//2)\n```'),
        '<think>Formula gives 5050.</think>\nThe answer is \\boxed{5050}',
        '<think>Output confirms it.</think>\nThe answer is \\boxed{5050}',
    ]

    class MockLLM:
        _c = 0

        def format_prompt(self, messages: List[Dict]) -> str:
            # Just concatenate for mock purposes
            return '\n'.join(m['content'] for m in messages)

        def generate(self, prompts: List[str], sp: SamplingParams) -> List[_RequestOutput]:
            out = []
            for _ in prompts:
                texts = [_MOCKS[(MockLLM._c + i) % len(_MOCKS)] for i in range(sp.n)]
                MockLLM._c += sp.n
                out.append(_RequestOutput(texts))
            return out

    llm = MockLLM()
    print('MockLLM ready.')


In [ ]:
# ============================================================
# Cell 6: TIR Inference with Self-Correction
# ============================================================

def _tir_continuation(
    messages: List[Dict],
    initial_response: str,
    llm_instance,
    max_rounds: int = MAX_TIR_ROUNDS,
) -> str:
    """
    Continue a TIR loop using proper chat message history.
    Executes code blocks and feeds output back to the model.
    Returns the extracted integer answer string, or ''.
    """
    sp1 = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P,
                         max_tokens=MAX_NEW_TOKENS, n=1)
    # Start conversation: [system, user, assistant(initial_response)]
    conv = messages + [{'role': 'assistant', 'content': initial_response}]

    for _ in range(max_rounds):
        last_assistant = conv[-1]['content']
        blocks = extract_code_blocks(last_assistant)
        if not blocks:
            break

        out_txt = execute_python(blocks[-1])
        conv.append({
            'role': 'user',
            'content': (
                f'Execution output:\n```\n{out_txt}\n```\n\n'
                'Continue your solution and provide the final answer.'
            ),
        })

        prompt = llm_instance.format_prompt(conv)
        new_text = llm_instance.generate([prompt], sp1)[0].outputs[0].text
        conv.append({'role': 'assistant', 'content': new_text})

        ans = extract_final_answer(new_text)
        if ans:
            return ans

    # Last resort: search all assistant turns
    full_text = ' '.join(
        m['content'] for m in conv if m['role'] == 'assistant'
    )
    return extract_final_answer(full_text) or ''


def tir_batch(problem: str, llm_instance, num_samples: int = NUM_SAMPLES) -> List[str]:
    """
    Generate num_samples independent solutions and return their answers.
    Each sample is a separate chain-of-thought with optional TIR loop.
    """
    messages = build_messages(problem)
    prompt   = llm_instance.format_prompt(messages)
    sp       = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P,
                              max_tokens=MAX_NEW_TOKENS, n=num_samples)
    batch    = llm_instance.generate([prompt], sp)

    answers: List[str] = []
    for s in batch[0].outputs:
        text = s.text
        ans  = extract_final_answer(text)

        if ans:
            answers.append(ans)
            continue

        if extract_code_blocks(text):
            # TIR: code block present but no answer yet → continue loop
            ans = _tir_continuation(messages, text, llm_instance)
        else:
            # Nudge: no code, no answer → force boxed answer
            nudge_messages = messages + [
                {'role': 'assistant', 'content': text},
                {'role': 'user',      'content':
                 'Please state your final integer answer as: The answer is \\boxed{N}'},
            ]
            nudge_prompt = llm_instance.format_prompt(nudge_messages)
            sp_n  = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=256, n=1)
            cont  = llm_instance.generate([nudge_prompt], sp_n)[0].outputs[0].text
            ans   = extract_final_answer(cont) or ''

        answers.append(ans)
    return answers


In [ ]:
# ============================================================
# Cell 7: Self-Consistency (Majority Voting)
# ============================================================

def majority_vote(answers: List[str]) -> str:
    valid = [a for a in answers if a.strip()]
    if not valid:
        return '0'
    # Normalise: strip commas/spaces, convert to int string if possible
    normalised = []
    for a in valid:
        try:
            normalised.append(str(int(a.replace(',', '').strip())))
        except ValueError:
            normalised.append(a.strip())
    return Counter(normalised).most_common(1)[0][0]


def solve_problem(problem: str, llm_instance, num_samples: int = NUM_SAMPLES) -> dict:
    ans_list = tir_batch(problem, llm_instance, num_samples)
    return {
        'answer': majority_vote(ans_list),
        'vote_distribution': dict(Counter(a for a in ans_list if a)),
        'num_valid': sum(1 for a in ans_list if a),
    }


print('Voting helpers OK')


In [ ]:
# ============================================================
# Cell 8: Main Pipeline
# ============================================================

def run_pipeline(
    llm_instance,
    test_csv: str  = TEST_CSV,
    output_csv: str = OUTPUT_CSV,
    num_samples: int = NUM_SAMPLES,
) -> pd.DataFrame:
    df = pd.read_csv(test_csv)
    assert {'id', 'problem'}.issubset(df.columns), f'Bad columns: {list(df.columns)}'

    rows, t0 = [], time.time()
    for i, (_, row) in enumerate(df.iterrows()):
        print(f"\n{'='*60}\n[{i+1}/{len(df)}] id={row['id']}")
        print(f"Problem : {str(row['problem'])[:120]}")
        ts  = time.time()
        res = solve_problem(row['problem'], llm_instance, num_samples)
        print(
            f"Answer: {res['answer']}  "
            f"Votes: {res['vote_distribution']}  "
            f"Valid: {res['num_valid']}/{num_samples}  "
            f"{time.time()-ts:.1f}s"
        )
        rows.append({'id': row['id'], 'answer': res['answer']})

    sub = pd.DataFrame(rows)
    sub['answer'] = pd.to_numeric(sub['answer'], errors='coerce').fillna(0).astype(int)
    # Clamp to non-negative (competition answers are non-negative integers)
    sub['answer'] = sub['answer'].clip(lower=0)
    sub.to_csv(output_csv, index=False)

    elapsed = (time.time() - t0) / 60
    print(f"\nDone: {len(df)} problems in {elapsed:.1f} min -> {output_csv}")
    print(sub.to_string(index=False))
    return sub


In [ ]:
# ============================================================
# Cell 9: Local Dummy Test  (skipped on Kaggle)
# ============================================================

if not IS_KAGGLE:
    print('>>> Local pipeline test with MockLLM <<<')
    dummy = [
        {'id': 1, 'problem': 'What is the sum of integers from 1 to 100?'},
        {'id': 2, 'problem': 'What is 2^10 mod 7?'},
        {'id': 3, 'problem': 'How many primes are less than 20?'},
    ]
    pd.DataFrame(dummy).to_csv(TEST_CSV, index=False)
    sub = run_pipeline(llm, test_csv=TEST_CSV, output_csv=OUTPUT_CSV)
    res = pd.read_csv(OUTPUT_CSV)
    assert list(res.columns) == ['id', 'answer'], f'Bad columns: {list(res.columns)}'
    assert len(res) == 3, 'Row count mismatch'
    assert pd.api.types.is_integer_dtype(res['answer']), f"Not int: {res['answer'].dtype}"
    print('\n>>> All assertions passed. <<<')


In [ ]:
# ============================================================
# Cell 10: Run on Kaggle
# ============================================================

if IS_KAGGLE:
    submission = run_pipeline(llm)
